#   초기세팅
    - 종목코드 설정
    - 분석 경로 설정
    - 병렬 처리

In [1]:
import os
import multiprocessing
import pandas as pd
import numpy as np
from pathlib import Path
from joblib import Parallel, delayed

#   종목 코드 지정
code_001 = '005930'

#   경로 설정
path_001 = './'

#   병렬처리 설정
cl = multiprocessing.cpu_count() - 1

#   경로 내 지정 종목코드에 대한 parquet파일 확인
df_list_001 = list(Path(path_001).rglob(f'*{code_001}*.parquet'))

#   pickle 파일 불러오기 함수 생성
def read_parquet_001(file_001):
    return pd.read_parquet(file_001)

In [ ]:
#   경로 내 지정 파일 불러오기
df_ls_001 = Parallel(n_jobs = -1)(delayed(read_parquet_001)(file_01) for file_01 in df_list_001)

In [ ]:
#   데이터 프레임으로 변환
df_001 = pd.concat(df_ls_001, ignore_index = True, axis = 0)
df_001.info(), df_001['answer'].value_counts(dropna=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 651361 entries, 0 to 651360
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   섹션           651361 non-null  int64         
 1   제목           651361 non-null  object        
 2   언론사          651361 non-null  object        
 3   본문           651361 non-null  object        
 4   Target_Date  651361 non-null  datetime64[ns]
 5   수정           651361 non-null  object        
 6   tokens       651361 non-null  object        
 7   answer       651361 non-null  bool          
dtypes: bool(1), datetime64[ns](1), int64(1), object(5)
memory usage: 35.4+ MB


(None,
 answer
 True     417997
 False    233364
 Name: count, dtype: int64)

In [ ]:
"""결측 및 중복 확인"""
#   데이터 내 결측 확인
for col_01 in df_001.columns.tolist():
    print(df_001.isnull()[col_01].value_counts().sort_index())

#   결측 제거
#df_001.dropna(subset = [''])

섹션
False    651361
Name: count, dtype: int64
제목
False    651361
Name: count, dtype: int64
언론사
False    651361
Name: count, dtype: int64
본문
False    651361
Name: count, dtype: int64
Target_Date
False    651361
Name: count, dtype: int64
수정
False    651361
Name: count, dtype: int64
tokens
False    651361
Name: count, dtype: int64
answer
False    651361
Name: count, dtype: int64


In [ ]:
#   '제목' 및 'Target_Date'열 기준, 중복 확인
df_001[['제목', 'Target_Date']].duplicated().value_counts()

#   중복 제거
#df_001.drop_duplicates(subset = ['제목', 'Target_Date'], inplace = True)

False    651361
Name: count, dtype: int64

##  데이터 추출 및 분리

In [ ]:
#   데이터 내 열 확인
df_001.columns.tolist()

#   'answer' 열 빈도 확인
df_001['answer'].value_counts()

answer
True     417997
False    233364
Name: count, dtype: int64

In [ ]:
#   입력 데이터 및 정답 데이터 추출
X = list(df_001['tokens'])
y = list(df_001['answer'])

len(X), len(y)

(651361, 651361)

In [ ]:
# 학습 데이터 및 테스트 데이터 분리
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)

len(X_train), len(X_test), len(y_train), len(y_test) 

(521088, 130273, 521088, 130273)

##   학습데이터 준비
### tokenizer 생성(Integer Encoding 용도)

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer_001 = Tokenizer()

#   numpy array들을 리스트(텍스트) 형태로 변환
tokenizer_001.fit_on_texts([item.tolist() if isinstance(item, np.ndarray) else item for item in X_train])

In [ ]:
#   단어 갯수 확인
len(tokenizer_001.word_index)

277765

In [ ]:
from review.감성분석_260523_analyzer_add_cnt_word import cnt_word

#   단어 등장빈도 설정
threshold = 25

#   threshold 이하 등장 단어 갯수 및 빈도 확인
cnt_word(tokenizer_001.word_counts.items(), threshold)


전체 단어 :  277,765개  113,287,344번 등장
제외 단어 (등장빈도 25이하) :  230,922개  967,419번 등장
제외 단어 비율 : 단어 빈도 83.14%/등장 빈도 0.85%
제외 단어 반영 : 46,843개 99.15%


In [ ]:
#   분석용 Tokenizer 생성
#   기준: 단어 출현 빈도 26이상 단어
num_vocab = 45289
num_words = num_vocab + 1#  + 1: index 0: padding 용도

tokenizer_002 = Tokenizer(num_words = num_words)
tokenizer_002.fit_on_texts([item.tolist() if isinstance(item, np.ndarray) else item for item in X_train])

### 입력 데이터 integer Encoding
    - 제한된 단어에만 index 부여
    - 희귀 단어로만 구성된 article: 단어 0(결측 해당)

In [ ]:
encoded_X_train = tokenizer_002.texts_to_sequences([item.tolist() if isinstance(item, np.ndarray) else item for item in X_train])

In [ ]:
#   길이 0 index 추출
null_index = [index for index, article in enumerate(encoded_X_train) if len(article) < 1]

In [ ]:
#   길이 0 기사 → 재구성(길이 1)
new_X_train = [article for index, article in enumerate(encoded_X_train) if index not in null_index]
new_y_train = [label for index, label in enumerate(y_train) if index not in null_index]

####    입력 데이터 padding
    - 입력 데이터 길이(max_len) 설정 후 padding

In [ ]:
#   기사 길이 분포 확인
len_df = pd.DataFrame([len(article) for article in new_X_train])

#   기술통계 확인
len_df.describe()

,0
count,519882.000000
mean,215.969887
std,137.470815
min,1.000000
25%,131.000000
50%,194.000000
75%,275.000000
max,5299.000000


In [ ]:
from mylib.my_utils import below_threshold_len_from_list
#   길이 max_len 이하 데이터 비중 확인
max_len = 700

below_threshold_len_from_list(max_len, new_X_train)

길이가 700 이하인 text의 비율 : 99.21%


In [ ]:
#   데이터 padding(기준: max_len 길이)
from tensorflow.keras.preprocessing.sequence import pad_sequences

#   신경망 입력 데이터 준비
input_X_train = pad_sequences(new_X_train, maxlen = max_len)

### 정답 데이터 one-hot encoding

In [ ]:
from tensorflow.keras.utils import to_categorical

input_y_train = to_categorical(new_y_train)

In [ ]:
import os
import glob

# './model/' 경로 내 h5 파일 목록 확인
model_dir = './model/'
model_files = glob.glob(os.path.join(model_dir, 'best_model_article_*.h5'))
print(model_files)

['./model\\best_model_article_260521.h5']


In [ ]:
model_files.sort()

latest_model_path = model_files[-1]

##  기존 학습 모델 불러오기 및 컴파일

In [ ]:
#   모델 설계
from tensorflow.keras.models import load_model
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint#   Embedding layer 파라미터

#   기존 학습 모델 불러오기
model = load_model(latest_model_path)

In [ ]:
#   미세 조정: 낮은 학습률 적용(0.0001)
model.compile(loss = 'binary_crossentropy', metrics = 'accuracy', optimizer=RMSprop(learning_rate=0.0001))

#   콜백 함수 재설정
es = EarlyStopping(monitor='val_loss', mode='min', patience=3, verbose=1)

In [ ]:
# 최대 날짜 추출
max_date = pd.to_datetime(df_001['Target_Date']).max().strftime('%y%m%d')

#   파일명 설정
mc = ModelCheckpoint(os.path.join(model_dir, f'best_model_article_{max_date}.h5'),
                      monitor='val_loss', mode='min', save_best_only=True)

In [ ]:
#   갱신 데이터 대상 학습 진행
model.fit(
    input_X_train,       
    input_y_train,       
    epochs=10,            
    batch_size=128, 
    validation_split=0.1, 
    callbacks=[es, mc]
)

Epoch 1/10
3656/3656 [==============================] - 1062s 289ms/step - loss: 0.6490 - accuracy: 0.6426 - val_loss: 0.6385 - val_accuracy: 0.6512
Epoch 2/10
3656/3656 [==============================] - 1146s 313ms/step - loss: 0.6255 - accuracy: 0.6613 - val_loss: 0.6180 - val_accuracy: 0.6701
Epoch 3/10
3656/3656 [==============================] - 1143s 313ms/step - loss: 0.6041 - accuracy: 0.6796 - val_loss: 0.6037 - val_accuracy: 0.6818
Epoch 4/10
3656/3656 [==============================] - 1172s 321ms/step - loss: 0.5870 - accuracy: 0.6938 - val_loss: 0.5922 - val_accuracy: 0.6886
Epoch 5/10
3656/3656 [==============================] - 928s 254ms/step - loss: 0.5729 - accuracy: 0.7043 - val_loss: 0.5912 - val_accuracy: 0.6938
Epoch 6/10
3656/3656 [==============================] - 1111s 304ms/step - loss: 0.5612 - accuracy: 0.7132 - val_loss: 0.5755 - val_accuracy: 0.7004
Epoch 7/10
3656/3656 [==============================] - 1150s 314ms/step - loss: 0.5506 - accuracy: 0.7197 

In [ ]:
#   테스트 데이터 Integer Encoding
encoded_X_test = tokenizer_002.texts_to_sequences([item.tolist() if isinstance(item, np.ndarray) else item for item in X_test])

#   빈 기사 처리: 길이 0 이하 
null_index_test = [index for index, article in enumerate(encoded_X_test) if len(article) < 1]

new_X_test = [article for index, article in enumerate(encoded_X_test) if index not in null_index_test]
new_y_test = [label for index, label in enumerate(y_test) if index not in null_index_test]

#   테스트 데이터 padding
input_X_test = pad_sequences(new_X_test, maxlen=max_len)

#   정답 데이터 원-핫 인코딩
input_y_test = to_categorical(new_y_test)

In [27]:
import numpy as np

#   학습 결과 저장
model.save(os.path.join(model_dir, f'model_final_article_{max_date}.h5'))

#   입력 데이터 저장
np.save('./data/article_X_test.npy', input_X_test)
np.save('./data/article_y_test.npy', input_y_test)